In [458]:
import re
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

# Задание 1

In [459]:
class Account:
    count = 1000
    transfer_count = 1

    account_type = 'checking'
    operations = ['deposit', 'withdraw']

	# Проверяет данные и создает счет
    def __init__(self, name, balance=0):
        pattern = (r'[А-Я][а-я]+ [А-Я][а-я]+'
                   r'|[A-Z][a-z]+ [A-Z][a-z]+')
        if not isinstance(name, str) or not re.fullmatch(pattern, name):
            raise ValueError('Имя должно быть в формате Имя Фамилия')
        if (not isinstance(balance, (int, float))
                or isinstance(balance, bool)):
            raise ValueError('Баланс должен быть числом')
        if not 0 <= balance < float('inf'):
            raise ValueError('Баланс должен быть неотрицательным числом')

        self.name = name
        self.balance = balance
        self.number = f'ACC-{Account.count}'
        Account.count += 1
        self.history = []

	# Показывает номер, владельца и баланс
    def __str__(self):
        return (f'{self.number}, {self.name}, {self.account_type}: '
                f'{self.balance:.2f} руб.')

	# Проверяет, что сумма положительная
    def check_amount(self, amount):
        if (not isinstance(amount, (int, float))
                or isinstance(amount, bool)):
            raise ValueError('Сумма должна быть числом')
        if not 0 < amount < float('inf'):
            raise ValueError('Сумма должна быть положительным числом')

	# Добавляет операцию в историю
    def add_record(self, operation, amount, status, date=None):
        if date is None:
            date = datetime.now()
        self.history.append({
            'operation': operation,
            'amount': amount,
            'date': date,
            'balance_after': self.balance,
            'status': status,
            'transfer': None,
        })

	# Пополняет счет
    def deposit(self, amount):
        self.check_amount(amount)
        self.balance = round(self.balance + amount, 2)
        self.add_record('deposit', amount, 'success')

	# Проверяет, хватает ли денег для снятия
    def can_withdraw(self, amount):
        return amount <= self.balance

	# Снимает деньги или записывает отказ
    def withdraw(self, amount):
        self.check_amount(amount)
        if not self.can_withdraw(amount):
            self.add_record('withdraw', amount, 'fail')
            return False

        self.balance = round(self.balance - amount, 2)
        self.add_record('withdraw', amount, 'success')
        return True

	#Возвращает историю в виде таблицы
    def get_history(self):
        columns = ['operation', 'amount', 'date', 'balance_after',
                   'status', 'transfer']
        return pd.DataFrame(self.history, columns=columns)

	# Выбирает последние крупные успешные операции
    def big_operations(self, amount, n):
        self.check_amount(amount)
        if not isinstance(n, int) or isinstance(n, bool) or n <= 0:
            raise ValueError('N должно быть положительным целым числом')
        data = self.get_history()
        data = data[(data['status'] == 'success')
                    & (data['amount'] >= amount)]
        return data.sort_values('date', kind='stable').tail(n)

	# Переводит деньги на другой счет
    def transfer(self, other, amount):
        if not isinstance(other, Account) or other is self:
            raise ValueError('Нужно выбрать другой счет')

        result = self.withdraw(amount)
        number = f'TR-{Account.transfer_count}'
        Account.transfer_count += 1
        self.history[-1]['transfer'] = number

        if result:
            other.deposit(amount)
        else:
            other.add_record('deposit', amount, 'fail')
        other.history[-1]['transfer'] = number
        return result


In [460]:
a = Account('Иван Иванов', 1000)
print(a)

a.deposit(500)
a.deposit(200)
a.withdraw(300)
a.withdraw(100)
print('Снятие 5000:', a.withdraw(5000))

print('Текущий баланс:', a.balance)
a.get_history()


ACC-1000, Иван Иванов, checking: 1000.00 руб.
Снятие 5000: False
Текущий баланс: 1300


,operation,amount,date,balance_after,status,transfer
0,deposit,500,2026-10-22 20:47:33.498496,1500,success,None
1,deposit,200,2026-10-22 20:47:33.498605,1700,success,None
2,withdraw,300,2026-10-22 20:47:33.498631,1400,success,None
3,withdraw,100,2026-10-22 20:47:33.498702,1300,success,None
4,withdraw,5000,2026-10-22 20:47:33.498734,1300,fail,None


# Задание 2

In [461]:
# Сберегательный счет с процентами и лимитом снятия
class SavingsAccount(Account):
    account_type = 'savings'
    operations = ['deposit', 'withdraw', 'interest']

	# Разрешает снять не больше половины баланса
    def can_withdraw(self, amount):
        return amount <= self.balance / 2

	# Начисляет проценты на текущий баланс
    def add_interest(self, rate):
        self.check_amount(rate)
        amount = round(self.balance * rate / 100, 2)
        if amount == 0:
            return
        self.balance = round(self.balance + amount, 2)
        self.add_record('interest', amount, 'success')


In [462]:
b = SavingsAccount('Анна Петрова', 2000)
print(a)
print(b)

b.deposit(1000)
b.add_interest(7)
print('После начисления процентов:', b.balance)
print('Снятие 2000:', b.withdraw(2000))
print('Снятие 500:', b.withdraw(500))
print('Текущий баланс:', b.balance)
b.get_history()


ACC-1000, Иван Иванов, checking: 1300.00 руб.
ACC-1001, Анна Петрова, savings: 2000.00 руб.
После начисления процентов: 3210.0
Снятие 2000: False
Снятие 500: True
Текущий баланс: 2710.0


,operation,amount,date,balance_after,status,transfer
0,deposit,1000.0,2026-10-22 20:47:33.537793,3000.0,success,None
1,interest,210.0,2026-10-22 20:47:33.537997,3210.0,success,None
2,withdraw,2000.0,2026-10-22 20:47:33.538090,3210.0,fail,None
3,withdraw,500.0,2026-10-22 20:47:33.538149,2710.0,success,None


In [463]:
a.big_operations(200, 2)


,operation,amount,date,balance_after,status,transfer
1,deposit,200,2026-10-22 20:47:33.498605,1700,success,None
2,withdraw,300,2026-10-22 20:47:33.498631,1400,success,None


In [464]:
try:
    Account('иван иванов')
except ValueError as e:
    print(e)

try:
    Account('Иван Иванов', -100)
except ValueError as e:
    print(e)

for amount in [0, -100, 'сто']:
    try:
        a.deposit(amount)
    except ValueError as e:
        print(e)


Имя должно быть в формате Имя Фамилия
Баланс должен быть неотрицательным числом
Сумма должна быть положительным числом
Сумма должна быть положительным числом
Сумма должна быть числом


### Доп задание

In [465]:
print('Успешный перевод:', a.transfer(b, 300))
print(a)
print(b)

print('Перевод сверх половины баланса:', b.transfer(a, 2000))
print('Перевод при нехватке денег:', a.transfer(b, 5000))
print(a)
print(b)


Успешный перевод: True
ACC-1000, Иван Иванов, checking: 1000.00 руб.
ACC-1001, Анна Петрова, savings: 3010.00 руб.
Перевод сверх половины баланса: False
Перевод при нехватке денег: False
ACC-1000, Иван Иванов, checking: 1000.00 руб.
ACC-1001, Анна Петрова, savings: 3010.00 руб.


In [466]:
a.get_history().tail(3)


,operation,amount,date,balance_after,status,transfer
5,withdraw,300,2026-10-22 20:47:33.577544,1000,success,TR-1
6,deposit,2000,2026-10-22 20:47:33.578047,1000,fail,TR-2
7,withdraw,5000,2026-10-22 20:47:33.578108,1000,fail,TR-3


In [467]:
b.get_history().tail(3)


,operation,amount,date,balance_after,status,transfer
4,deposit,300.0,2026-10-22 20:47:33.577571,3010.0,success,TR-1
5,withdraw,2000.0,2026-10-22 20:47:33.578041,3010.0,fail,TR-2
6,deposit,5000.0,2026-10-22 20:47:33.578111,3010.0,fail,TR-3


# Задание 3

In [468]:
# Загружает операции счета и показывает ошибки в данных
def load_csv(account, file, number):
    data = pd.read_csv(file)
    total = len(data)
    data = data[data['account_number'] == number].copy()
    other = total - len(data)
    data['repeat'] = data.duplicated()
    errors = []
    count = 0

    for i in data.index:
        row = data.loc[i]
        try:
            if row['repeat']:
                raise ValueError('Повтор строки')
            if row['account_type'] != account.account_type:
                raise ValueError('Неверный тип счета')
            if row['operation'] not in account.operations:
                raise ValueError('Недопустимая операция для этого счета')
            if row['status'] not in ['success', 'fail']:
                raise ValueError('Неверный статус')

            try:
                amount = float(row['amount'])
            except (ValueError, TypeError):
                raise ValueError('Сумма не является числом')
            account.check_amount(amount)

            date = None
            for fmt in ['%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M']:
                try:
                    date = datetime.strptime(str(row['date']), fmt)
                    break
                except ValueError:
                    pass
            if date is None:
                raise ValueError('Некорректная дата')

            if row['status'] == 'success':
                if row['operation'] == 'withdraw':
                    if not account.can_withdraw(amount):
                        raise ValueError('Снятие нарушает правила счета')
                    account.balance = round(account.balance - amount, 2)
                else:
                    account.balance = round(account.balance + amount, 2)

            account.add_record(row['operation'], amount, row['status'], date)
            count += 1
        except ValueError as e:
            errors.append({'row': i + 2, 'reason': str(e)})

    print('Принято:', count)
    print('Отклонено:', len(errors))
    print('Строк других счетов:', other)
    return pd.DataFrame(errors, columns=['row', 'reason'])


### Загрузка в расчетный счет

In [469]:
file = 'Файлы к ДЗ 5/transactions_dirty.csv'
c = Account('Петр Смирнов', 1200)
errors_c = load_csv(c, file, 'ACC-100001')
print(c)
errors_c


Принято: 17
Отклонено: 9
Строк других счетов: 79
ACC-1002, Петр Смирнов, checking: 4864.00 руб.


,row,reason
0,7,Сумма должна быть положительным числом
1,10,Недопустимая операция для этого счета
2,12,Недопустимая операция для этого счета
3,15,Некорректная дата
4,17,Недопустимая операция для этого счета
5,18,Недопустимая операция для этого счета
6,19,Некорректная дата
7,26,Неверный статус
8,106,Повтор строки


In [470]:
c.get_history().tail()


,operation,amount,date,balance_after,status,transfer
12,deposit,532.0,2025-10-14 22:17:26,4729.0,success,None
13,withdraw,393.0,2025-10-16 22:17:26,4336.0,success,None
14,deposit,701.0,2025-10-16 22:17:26,5037.0,success,None
15,withdraw,308.0,2025-10-17 22:17:26,4729.0,success,None
16,deposit,135.0,2025-10-18 22:17:26,4864.0,success,None


### Загрузка в сберегательный счет

In [471]:
d = SavingsAccount('Мария Соколова', 1500)
errors_d = load_csv(d, file, 'ACC-100002')
print(d)
errors_d


Принято: 18
Отклонено: 9
Строк других счетов: 78
ACC-1003, Мария Соколова, savings: 3991.24 руб.


,row,reason
0,30,Некорректная дата
1,31,Сумма должна быть положительным числом
2,35,Сумма должна быть положительным числом
3,39,Недопустимая операция для этого счета
4,46,Сумма должна быть положительным числом
5,48,Неверный статус
6,50,Недопустимая операция для этого счета
7,102,Повтор строки
8,105,Повтор строки


In [472]:
d.get_history().tail()


,operation,amount,date,balance_after,status,transfer
13,withdraw,231.00,2025-10-14 22:17:26,3982.98,success,None
14,interest,274.13,2025-10-15 22:17:26,4257.11,success,None
15,withdraw,286.00,2025-10-17 22:17:26,3971.11,success,None
16,interest,302.13,2025-10-20 22:17:26,4273.24,success,None
17,withdraw,282.00,2025-10-23 22:17:26,3991.24,success,None
